# Architecture and task validation

This is an interface-validation report, not a model-selection benchmark.
Small training budgets exercise public methods across the registry. Metrics
are labeled by task and evaluation split; poor scores are not teaching targets.
For fitted teaching examples, use the [tutorials](../../tutorials/index.rst).


**Execution environment:** nampy 0.3.0, numpy 1.26.4, scikit-learn 1.9.0, torch 2.14.1+cpu, lightning 2.6.6. Python 3.12.13; source base commit `ecdb0b1` (working checkout); elapsed 86.0 seconds on this CPU. Runtime varies by hardware.

## 1. Data and model setup

The same synthetic covariates are used throughout so differences in the
returned predictions, scores, and additive components reflect the modeled
quantity rather than an unrelated change of dataset.


In [1]:
from pathlib import Path
import tempfile
import time
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#F97316", "#10B981", "#8B5CF6", "#EF4444"]

N_SAMPLES = 1000
MAX_EPOCHS = 20
RANDOM_STATE = 7
CHECKPOINT_ROOT = Path(tempfile.mkdtemp(prefix="nampy-notebook-"))

NEURAL_FIT_KWARGS = {
    "max_epochs": MAX_EPOCHS,
    "batch_size": 256,
    "random_state": RANDOM_STATE,
    "checkpoint_path": CHECKPOINT_ROOT,
    "logger": False,
    "enable_progress_bar": False,
    "enable_model_summary": False,
    "num_sanity_val_steps": 0,
}

rng = np.random.default_rng(RANDOM_STATE)

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from nampy import models
from nampy.neural.registry import architectures

n = N_SAMPLES
x = np.linspace(0.03, 0.97, n)
groups = pd.Categorical(np.resize(np.array(["a", "b", "c"]), n))
X = pd.DataFrame({"x1": x, "x2": np.cos(2 * np.pi * x), "group": groups})
y_reg = np.sin(2 * np.pi * x) + 0.25 * x + 0.2 * (groups == "b")
y_cls = np.resize(np.array([0, 1]), n)
y_lss = y_reg + np.exp(-1.2 + 0.8 * x) * rng.normal(size=n)

SWEEP_FIT_KWARGS = {
    **NEURAL_FIT_KWARGS,
    "batch_size": 256,
    "shuffle": False,
}

MODEL_KWARGS = {
    "linreg": {},
    "nam": {"layer_sizes": [8], "dropout": 0.0},
    "sian": {"layer_sizes": [8], "interactions": [("x1", "x2")]},
    "snam": {"layer_sizes": [8], "dropout": 0.0, "group_lasso_lambda": 0.01},
    "gpnam": {"rff_num_feat": 8},
    "igann": {"n_hid": 4, "n_estimators": 3, "early_stopping": 2},
    "nbm": {"layer_sizes": [8], "num_bases": 4, "dropout": 0.0, "bases_dropout": 0.0, "output_penalty": 0.01},
    "nbm_spam": {"layer_sizes": [8], "num_bases": 4, "ranks": [4], "batch_norm": False},
    "spam": {"ranks": [4], "dropout": 0.0},
    "natt": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "namformer": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "treenam": {"tree_depth": 2, "tree_lamda": 0.01},
    "ensemble_treenam": {"tree_depth": 2, "tree_lamda": 0.01, "num_estimators": 2},
    "nodegam": {"num_trees": 4, "num_layers": 1, "depth": 2, "last_dropout": 0.0, "anneal_steps": 10, "interaction_degree": 1},
    "qnam": {"layer_sizes": [8], "dropout": 0.0, "distributional_kwargs": {"quantiles": [0.1, 0.5, 0.9]}},
    "spline_nam": {"n_knots": 5, "smoothing": 0.01},
}

def estimator_class(spec, suffix):
    return getattr(models, f"{spec.estimator_prefix}{suffix}", None)

def make_estimator(name, spec, suffix, **extra):
    cls = estimator_class(spec, suffix)
    if cls is None:
        return None
    kwargs = dict(MODEL_KWARGS[name])
    kwargs.update(extra)
    return cls(**kwargs)

def reconstruction_error(estimator, features):
    components = estimator.predict_components(features, center=True)
    link = np.asarray(components.link)
    def aligned(value):
        array = np.asarray(value, dtype=float)
        if link.ndim == 1 and array.shape == (link.shape[0], 1):
            return array[:, 0]
        if link.ndim == 2 and array.shape == (link.shape[0],):
            return array[:, None]
        return array
    reconstruction = aligned(components.intercept)
    for contribution in components.terms.values():
        reconstruction = reconstruction + aligned(contribution)
    if components.offset is not None:
        reconstruction = reconstruction + aligned(components.offset)
    error = float(np.max(np.abs(reconstruction - link)))
    valid = error <= 1e-5
    assert valid, (type(estimator).__name__, error)
    return error, valid


## 2. Training-data regression checks

Every registered regression architecture is fitted on the same rows. The
score, runtime, additive reconstruction, and term importance are recorded.
Architecture-specific output is added where available. This confirms the
installed runtime rather than relying on a copied documentation matrix.

Under squared-error loss, the optimal point predictor is the conditional
mean $\mathbb E[Y\mid x]$. A good mean prediction does not describe how
widely outcomes vary around it. Read the regression scores as measures of
point prediction; later sections add likelihood and interval checks for the
rest of the conditional distribution.


In [2]:
regression_models = {}
regression_rows = []
for name, spec in architectures().items():
    if not spec.supports("regression"):
        continue
    estimator = make_estimator(name, spec, "Regressor")
    started = time.perf_counter()
    estimator.fit(X, y_reg, **SWEEP_FIT_KWARGS)
    importance = estimator.term_importance(X, center=True)
    error, reconstruction_valid = reconstruction_error(estimator, X)
    row = {
        "architecture": name,
        "status": "Checked successfully",
        "R2": estimator.score(X, y_reg),
        "seconds": time.perf_counter() - started,
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "most important term": importance.iloc[0]["term"] if len(importance) else None,
    }
    if hasattr(estimator, "local_term_importance") and spec.supports("local_term_importance"):
        row["special output"] = "local importance"
    elif name == "igann":
        row["special output"] = f"{estimator.n_estimators_} boosting stages"
    else:
        row["special output"] = None
    regression_models[name] = estimator
    regression_rows.append(row)

regression_report = pd.DataFrame(regression_rows)
regression_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,architecture,status,R2,seconds,reconstruction error,reconstruction valid,most important term,special output
0,linreg,Checked successfully,-0.416730,1.111854,1.150183e-07,True,group,NaN
1,nam,Checked successfully,-0.703698,1.139223,2.421439e-08,True,x1,NaN
2,sian,Checked successfully,0.489438,0.969360,7.264316e-08,True,x1:x2,NaN
3,snam,Checked successfully,-0.319681,1.094064,2.747402e-08,True,group,NaN
4,gpnam,Checked successfully,0.024490,0.112796,2.043089e-08,True,group[1],NaN
5,igann,Checked successfully,0.324776,0.110897,3.911555e-08,True,group,3 boosting stages
6,nbm,Checked successfully,-0.095009,1.146359,1.396984e-08,True,group[1],NaN
7,nbm_spam,Checked successfully,-0.004501,1.326208,1.629815e-08,True,group[0],NaN
8,spam,Checked successfully,-0.177864,0.925978,5.541369e-08,True,group[0],local importance
9,natt,Checked successfully,0.819874,1.901234,7.380731e-08,True,x1,NaN


## 3. Training-data classification checks

Binary classifier components reconstruct logits before the probability
transform. Multiclass prediction is supported where advertised, but neural
classifier component extraction is currently restricted to binary classification. Log loss evaluates probability quality,
ROC-AUC evaluates ranking for the positive class, accuracy evaluates hard
labels, and calibration error compares binned predicted and empirical
frequencies. Link-scale terms explain the model score—not an additive
decomposition of probability.

For binary labels, $p(x)=\operatorname{sigmoid}\{\eta(x)\}$ and the
negative log likelihood is

$$-\frac1n\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].$$

Log loss penalizes confident wrong probabilities. AUC depends on ordering,
so a model can rank well while reporting poorly calibrated probabilities.
Compare the probability metrics together rather than interpreting the
classification score as a complete assessment.


In [3]:
def expected_calibration_error(y_true, probability, bins=6):
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(probability, edges) - 1, 0, bins - 1)
    total = len(y_true)
    return sum(
        np.sum(index == b) / total * abs(np.mean(probability[index == b]) - np.mean(y_true[index == b]))
        for b in range(bins) if np.any(index == b)
    )

classification_models = {}
classification_rows = []
for name, spec in architectures().items():
    if not spec.supports("classification"):
        continue
    estimator = make_estimator(name, spec, "Classifier")
    started = time.perf_counter()
    estimator.fit(X, y_cls, **SWEEP_FIT_KWARGS)
    probability = estimator.predict_proba(X)[:, 1]
    label = estimator.predict(X)
    importance = estimator.term_importance(X, center=True)
    error, reconstruction_valid = reconstruction_error(estimator, X)
    classification_rows.append({
        "architecture": name, "status": "Checked successfully",
        "log loss": log_loss(y_cls, probability),
        "ROC-AUC": roc_auc_score(y_cls, probability),
        "accuracy": accuracy_score(y_cls, label),
        "calibration error": expected_calibration_error(y_cls, probability),
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "most important term": importance.iloc[0]["term"] if len(importance) else None,
        "seconds": time.perf_counter() - started,
    })
    classification_models[name] = estimator
classification_report = pd.DataFrame(classification_rows)
classification_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,architecture,status,log loss,ROC-AUC,accuracy,calibration error,reconstruction error,reconstruction valid,most important term,seconds
0,linreg,Checked successfully,0.720785,0.500800,0.500,0.096465,1.545995e-07,True,group,1.180164
1,nam,Checked successfully,0.703513,0.500876,0.500,0.065429,3.213063e-08,True,x1,0.990782
2,sian,Checked successfully,0.694319,0.498836,0.500,0.018074,3.259629e-08,True,x1:x2,0.755364
3,snam,Checked successfully,0.703215,0.500716,0.499,0.063022,2.793968e-08,True,group,1.113196
4,gpnam,Checked successfully,0.693391,0.500634,0.507,0.002121,1.536682e-08,True,group[0],0.888352
5,igann,Checked successfully,0.693474,0.501000,0.501,0.011553,3.725290e-09,True,group,0.095701
6,nbm,Checked successfully,0.697058,0.499310,0.500,0.038224,2.235174e-08,True,group[1],1.118909
7,nbm_spam,Checked successfully,0.693180,0.500406,0.500,0.003792,1.443550e-08,True,group[0],1.194740
8,spam,Checked successfully,0.695024,0.501568,0.500,0.026432,5.215406e-08,True,x1,0.947254
9,natt,Checked successfully,0.696564,0.500672,0.501,0.037526,1.565786e-08,True,group,1.822031


## 4. Training-data location-scale checks

Normal LSS models predict a location and positive scale. Each parameter
has its own link-scale additive predictor, so raw component reconstruction
is checked column by column. Mean negative log likelihood evaluates the
complete conditional distribution; interval coverage checks whether the
predicted scale is calibrated. QNAM is intentionally handled in the
quantile gallery because its fixed family is not normal.

A normal location-scale model has
$Y\mid x\sim\mathcal N\{\mu(x),\sigma(x)^2\}$, with separate additive
predictors transformed to a real location and positive scale. Its negative
log likelihood includes both $\log\sigma$ and the squared residual divided
by $2\sigma^2$. Thus scale must explain the size of residual variation,
not simply become large everywhere.

For a fitted normal distribution, $\mu\pm1.96\sigma$ is a nominal 95%
prediction interval. It describes future outcomes conditional on the fitted
parameters, not a confidence interval for the mean. Held-out coverage and
interval width help reveal whether the predicted spread is useful.


In [4]:
lss_models = {}
lss_rows = []
for name, spec in architectures().items():
    if not spec.supports("distributional") or name == "qnam":
        continue
    estimator = make_estimator(name, spec, "LSS", family="normal")
    started = time.perf_counter()
    estimator.fit(X, y_lss, **SWEEP_FIT_KWARGS)
    parameters = estimator.predict(X)
    raw = estimator.predict(X, raw=True)
    lower = parameters[:, 0] - 1.645 * parameters[:, 1]
    upper = parameters[:, 0] + 1.645 * parameters[:, 1]
    error, reconstruction_valid = reconstruction_error(estimator, X)
    lss_rows.append({
        "architecture": name, "status": "Checked successfully",
        "NLL": estimator.evaluate(X, y_lss)["NLL"],
        "minimum scale": parameters[:, 1].min(),
        "90% coverage": np.mean((y_lss >= lower) & (y_lss <= upper)),
        "raw width": raw.shape[1],
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "seconds": time.perf_counter() - started,
    })
    lss_models[name] = estimator
lss_report = pd.DataFrame(lss_rows)
lss_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,architecture,status,NLL,minimum scale,90% coverage,raw width,reconstruction error,reconstruction valid,seconds
0,linreg,Checked successfully,1.298374,0.491920,0.916,2,1.150183e-07,True,0.981768
1,nam,Checked successfully,1.351217,0.526378,0.784,2,3.911555e-08,True,0.892998
2,sian,Checked successfully,0.985271,0.530487,0.922,2,7.078052e-08,True,0.760216
3,snam,Checked successfully,1.248810,0.787541,0.921,2,5.261973e-08,True,1.062123
4,gpnam,Checked successfully,1.142782,0.496734,0.922,2,8.940697e-08,True,0.747302
5,igann,Checked successfully,0.898148,0.438979,0.903,2,1.890585e-07,True,0.763592
6,nbm,Checked successfully,1.222517,0.683150,0.869,2,2.072193e-08,True,0.884050
7,nbm_spam,Checked successfully,1.185842,0.701436,0.868,2,4.703179e-08,True,1.085281
8,spam,Checked successfully,1.295459,0.629661,0.837,2,7.590279e-08,True,0.736155
9,natt,Checked successfully,1.008232,0.431293,0.940,2,7.031485e-08,True,1.416526


## 5. Distribution families

A distribution family owns target support, output width, parameter
transforms, likelihood, point prediction, and default metrics. Continuous,
count, discrete/ordered, multivariate, and quantile families therefore
require different targets. The loop fits one small canonical NAMLSS model
per installed family. These fits test the interface and output domains;
they are not estimates worth interpreting scientifically.

Distributional regression specifies
$p\{y\mid\theta_1(x),\ldots,\theta_K(x)\}$ rather than only a mean.
Each parameter has a domain enforced by its transform. Counts, positive
continuous responses, and bounded outcomes call for different likelihoods;
a valid transform alone does not make a family suitable for the observed
data. Compare likelihood scores only when they refer to the same target
and a compatible probability measure.


In [5]:
from nampy.models import NAMLSS
from nampy.neural.distributions.registry import FAMILY_REGISTRY

positive = np.exp(0.2 + 0.4 * x + rng.normal(0, 0.15, n))
counts = rng.poisson(np.exp(0.2 + 0.5 * x))
zero_counts = counts.copy(); zero_counts[::3] = 0
classes3 = np.resize(np.arange(3), n)
composition = rng.dirichlet([2.0, 3.0, 4.0], size=n)
multi_y = np.column_stack([y_lss, np.cos(np.pi * x) + rng.normal(0, 0.1, n)])

family_targets = {
    "normal": y_lss, "poisson": counts, "gamma": positive,
    "beta": np.clip(1 / (1 + np.exp(-y_lss)), 1e-4, 1-1e-4),
    "dirichlet": composition, "studentt": y_lss,
    "negativebinom": counts, "inversegamma": positive,
    "categorical": classes3, "quantile": y_lss, "robustnormal": y_lss,
    "lognormal": positive, "weibull": positive, "loglogistic": positive,
    "zip": zero_counts, "zinb": zero_counts,
    "hurdlepoisson": zero_counts, "hurdlenegativebinom": zero_counts,
    "tweedie": zero_counts.astype(float), "ordinal": classes3,
    "mvnormdiag": multi_y,
}
family_groups = {
    "continuous": {"normal", "gamma", "beta", "studentt", "inversegamma", "robustnormal", "lognormal", "weibull", "loglogistic", "tweedie"},
    "counts": {"poisson", "negativebinom", "zip", "zinb", "hurdlepoisson", "hurdlenegativebinom"},
    "discrete and ordered": {"categorical", "ordinal"},
    "multivariate": {"dirichlet", "mvnormdiag"},
    "quantiles": {"quantile"},
}

def group_for_family(name):
    return next(group for group, names in family_groups.items() if name in names)

family_models = {}
family_rows = []
for name in FAMILY_REGISTRY:
    kwargs = {"layer_sizes": [8], "dropout": 0.0}
    if name == "quantile":
        kwargs["distributional_kwargs"] = {"quantiles": [0.1, 0.5, 0.9]}
    estimator = NAMLSS(family=name, **kwargs)
    target = family_targets[name]
    started = time.perf_counter()
    estimator.fit(X, target, **SWEEP_FIT_KWARGS)
    raw = estimator.predict(X, raw=True)
    natural = estimator.predict(X)
    metrics = estimator.evaluate(X, target)
    if name == "quantile":
        levels = np.asarray([0.1, 0.5, 0.9])
        residual = np.asarray(target)[:, None] - natural
        metric_name = "pinball loss"
        metric_value = float(np.maximum(levels * residual, (levels - 1) * residual).sum(axis=1).mean())
        assert np.isclose(metric_value, metrics["pinball_loss"], rtol=1e-5)
    else:
        metric_name, metric_value = "NLL", metrics["NLL"]
    assert np.isfinite(metric_value)
    family_rows.append({
        "group": group_for_family(name), "family": name,
        "status": "Checked successfully", "target shape": np.asarray(target).shape,
        "raw parameters": raw.shape[1], "finite output": bool(np.isfinite(natural).all()),
        "metric": metric_name, "metric value": metric_value,
        "seconds": time.perf_counter() - started,
    })
    family_models[name] = estimator
family_report = pd.DataFrame(family_rows).sort_values(["group", "family"])
family_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


,group,family,status,target shape,raw parameters,finite output,metric,metric value,seconds
3,continuous,beta,Checked successfully,"(1000,)",2,True,NLL,0.175786,0.789050
2,continuous,gamma,Checked successfully,"(1000,)",2,True,NLL,1.575774,1.013231
7,continuous,inversegamma,Checked successfully,"(1000,)",2,True,NLL,1.659127,0.948173
13,continuous,loglogistic,Checked successfully,"(1000,)",2,True,NLL,2.220873,0.804099
11,continuous,lognormal,Checked successfully,"(1000,)",2,True,NLL,0.950339,1.055665
0,continuous,normal,Checked successfully,"(1000,)",2,True,NLL,1.351217,0.962195
10,continuous,robustnormal,Checked successfully,"(1000,)",2,True,NLL,0.433843,0.835327
5,continuous,studentt,Checked successfully,"(1000,)",3,True,NLL,1.703900,0.852877
18,continuous,tweedie,Checked successfully,"(1000,)",2,True,NLL,2.525686,0.839133
12,continuous,weibull,Checked successfully,"(1000,)",2,True,NLL,1.909803,0.821335


### 5.1 Inspect parameter constraints

Natural transforms enforce positive rates/scales and unit-interval
probabilities. They do not guarantee calibration. QNAM additionally
enforces ordered quantiles numerically.

For quantile outputs, the pinball loss weights positive and negative
residuals asymmetrically to target a chosen probability level. An ordered
parameterization ensures $q_{\tau_1}(x)\leq q_{\tau_2}(x)$ when
$\tau_1<\tau_2$. The parameter checks below verify output domains and
ordering. Calibration is a separate empirical property that needs held-out
outcomes.

The quantile-family metric is pinball loss summed over levels and averaged
over observations, matching the training objective and ``evaluate``.


In [6]:
domain_checks = {
    "normal scale positive": bool((family_models["normal"].predict(X)[:, 1] > 0).all()),
    "beta parameters positive": bool((family_models["beta"].predict(X) > 0).all()),
    "ZIP finite": bool(np.isfinite(family_models["zip"].predict(X)).all()),
}

qnam = make_estimator("qnam", architectures()["qnam"], "LSS")
qnam.fit(X, y_lss, **SWEEP_FIT_KWARGS)
quantiles = qnam.predict(X)
minimum_quantile_gap = float(np.diff(quantiles, axis=1).min())
domain_checks["QNAM minimum adjacent quantile gap"] = minimum_quantile_gap

qnam.predict_components(X).validate_additive_reconstruction(rtol=1e-5, atol=1e-5)
quantile_verified = bool(np.isfinite(quantiles).all() and minimum_quantile_gap >= 0)
assert quantile_verified

domain_checks


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


{'normal scale positive': True,
 'beta parameters positive': True,
 'ZIP finite': True,
 'QNAM minimum adjacent quantile gap': 2.7959256172180176}

## 6. Ensemble uncertainty

`NeuralEnsemble` clones and fits regressors or classifiers independently.
Bootstrapping changes member datasets; random seeds change initialization
and data order; `n_jobs` controls parallel fitting. Predictions are
averaged on the response scale, while terms and links are averaged on the
additive scale. Persistence is inherited from the member estimators only;
the complete ensemble can be serialized with joblib.

For response predictions $\hat\mu_m(x)$, the ensemble prediction is
$\bar\mu(x)=M^{-1}\sum_m\hat\mu_m(x)$. Their sample standard deviation
measures disagreement among the fitted members. It does not include
observation noise unless that is modeled separately.

With a nonlinear inverse link, averaging response predictions is different
from transforming an average predictor:
$M^{-1}\sum_m g^{-1}(\eta_m)\ne g^{-1}(M^{-1}\sum_m\eta_m)$ in general.
The averaged components therefore explain the averaged link predictor;
do not automatically equate its inverse transform with the ensemble's
response-scale prediction.


In [7]:
from joblib import dump, load
from nampy.models import NAMRegressor, NeuralEnsemble

ensemble = NeuralEnsemble(
    NAMRegressor(layer_sizes=[8], dropout=0.0),
    n_estimators=3, random_state=RANDOM_STATE, n_jobs=1, bootstrap=True,
)
ensemble.fit(X, y_reg, **SWEEP_FIT_KWARGS)
ensemble_prediction = ensemble.predict(X)
ensemble_components = ensemble.predict_components(X, center=True)
uncertainty = ensemble.predict_component_uncertainty(X, center=True)
ensemble_reconstruction_error, _ = reconstruction_error(ensemble, X)

ensemble_path = CHECKPOINT_ROOT / "neural-ensemble.joblib"
dump(ensemble, ensemble_path)
restored_ensemble = load(ensemble_path)
reload_difference = float(np.max(np.abs(restored_ensemble.predict(X) - ensemble_prediction)))

display({
    "members": uncertainty.n_estimators,
    "mean response SD": float(np.mean(uncertainty.response_std)),
    "mean term SD": {name: float(values.mean()) for name, values in uncertainty.term_std.items()},
    "reconstruction error": ensemble_reconstruction_error,
    "maximum reload difference": reload_difference,
})


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=20` reached.


{'members': 3,
 'mean response SD': 0.1983688920736313,
 'mean term SD': {'x1': 0.04003573446966001,
  'x2': 0.12111195366401764,
  'group': 0.034382466944564045},
 'reconstruction error': 3.89603276862438e-08,
 'maximum reload difference': 0.0}

### 6.1 Three different meanings of uncertainty

- **Ensemble disagreement** measures variation among fitted neural members
  under seeds and/or bootstrap samples. It is not automatically a
  calibrated confidence interval.
- **GAM standard errors** arise from an approximate sampling covariance for
  fitted coefficients, conditional on model assumptions and optionally
  including smoothing uncertainty.
- **LSS conditional uncertainty** is the predicted response distribution
  for a new outcome given its features. It can be large even if parameter
  estimates are known precisely.


## 7. Supported tasks and checked outputs

**Supported** describes the declared interface, with restrictions in the
[model capability reference](../../models/index.rst). The table below reports
validation evidence, rather than repeating the registry as a test result.

- **Checked successfully:** the corresponding fit/output check ran and passed.
- **Not checked in this report:** support is declared, but no corresponding check passed here.
- **Unavailable:** the registry does not expose that task or interface.
- **Not applicable:** the property is not relevant to that architecture.

Task cells come from the regression, classification, and LSS reports above;
those runs check prediction outputs and additive reconstruction. Quantile
cells come from the ordered, finite-output and reconstruction checks in
section 5.1. The code below records the checks behind each row.

Term columns are checked only when fitted models return finite contributions
with that metadata kind and pass additive reconstruction at a $10^{-5}$
tolerance for float32 arithmetic. Main, contextual, global, and interaction
terms are checked separately. **Not returned in checked fits** describes the
observed components, rather than claiming the architecture cannot expose them.


In [8]:
def task_status(name, capability, report):
    spec = architectures()[name]
    if not spec.supports(capability):
        return "Unavailable"
    match = report.loc[report["architecture"] == name]
    if len(match) and match.iloc[0]["status"] == "Checked successfully":
        return "Checked successfully"
    return "Not checked in this report"

def demonstrated_term_kinds(name):
    fitted = [models[name] for models in [regression_models, classification_models, lss_models]
              if name in models]
    if name == "qnam" and quantile_verified:
        fitted.append(qnam)
    observed = set()
    for estimator in fitted:
        components = estimator.predict_components(X)
        assert np.isfinite(np.asarray(components.link)).all()
        components.validate_additive_reconstruction(rtol=1e-5, atol=1e-5)
        for term, contribution in components.terms.items():
            values = np.asarray(contribution)
            assert len(values) == len(X) and np.isfinite(values).all()
            metadata = components.term_metadata.get(term)
            if metadata is not None:
                observed.add(metadata.kind)
    return observed

capability_rows = []
for name, spec in architectures().items():
    observed = demonstrated_term_kinds(name)
    capability_rows.append({
        "model": name,
        "Task evidence": ", ".join(label for label, report in [("section 2: regression", regression_report), ("section 3: classification", classification_report), ("section 4: normal LSS", lss_report)] if name in set(report["architecture"])) if name != "qnam" else "section 5.1: ordered quantiles and reconstruction",
        "Term evidence": "finite predict_components values, metadata kinds, and reconstruction",
        "Regression": task_status(name, "regression", regression_report),
        "Classification": task_status(name, "classification", classification_report),
        "LSS": ("Checked successfully" if quantile_verified else "Not checked in this report") if name == "qnam" else task_status(name, "distributional", lss_report),
        **{label: "Checked successfully" if kind in observed else "Not returned in checked fits"
           for label, kind in [("Main effects", "main"), ("Contextual terms", "contextual"), ("Global terms", "global")]},
        "Interactions": "Checked successfully" if "interaction" in observed else ("Not checked in this report" if spec.supports("interactions") else "Unavailable"),
        "Higher-order interactions": "Not checked in this report" if name in {"sian", "nbm", "spam", "nbm_spam"} else "Not applicable",
        "Sparsity": "Not checked in this report" if name in {"snam", "sian", "nbm", "igann"} else "Not applicable",
        "Local importance": "Not checked in this report" if spec.supports("local_term_importance") else "Unavailable",
        "Quantiles": "Checked successfully" if name == "qnam" and quantile_verified else "Unavailable",
    })

final_capability_report = pd.DataFrame(capability_rows)
final_capability_report


,model,Task evidence,Term evidence,Regression,Classification,LSS,Main effects,Contextual terms,Global terms,Interactions,Higher-order interactions,Sparsity,Local importance,Quantiles
0,linreg,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Unavailable,Not applicable,Not applicable,Unavailable,Unavailable
1,nam,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Not checked in this report,Not applicable,Not applicable,Unavailable,Unavailable
2,sian,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Checked successfully,Not checked in this report,Not checked in this report,Unavailable,Unavailable
3,snam,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Not checked in this report,Not applicable,Not checked in this report,Unavailable,Unavailable
4,gpnam,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Not checked in this report,Not applicable,Not applicable,Unavailable,Unavailable
5,igann,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Unavailable,Not applicable,Not checked in this report,Unavailable,Unavailable
6,nbm,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Not checked in this report,Not checked in this report,Not checked in this report,Unavailable,Unavailable
7,nbm_spam,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Not checked in this report,Not checked in this report,Not applicable,Unavailable,Unavailable
8,spam,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not returned in checked fits,Not checked in this report,Not checked in this report,Not applicable,Not checked in this report,Unavailable
9,natt,"section 2: regression, section 3: classificati...","finite predict_components values, metadata kin...",Checked successfully,Checked successfully,Checked successfully,Checked successfully,Checked successfully,Not returned in checked fits,Not checked in this report,Not applicable,Not applicable,Unavailable,Unavailable


## 8. Interpreting the results

Successful execution verifies the estimator interface and output shapes;
it does not establish that an architecture is suitable for a particular
dataset. For a substantive comparison, choose metrics before fitting, tune
inside cross-validation, use repeated splits, inspect term stability, and
compare against simple baselines.
